# Project Overview

# Financial Risk RAG Assistant

## Objective
Build a Retrieval Augmented Generation (RAG) system to answer questions on financial risks using the annual reports (10k-filings).

## Key Features




*   Multi-document retrieval
*   Source-grounded answers
*   FAISS vector search
*   OpenAI answer generation
*   Citation tracking

## Tech Stack
Python
LangChain
FAISS
SentenceTransformers
OpenAI API

## Pipeline Architecture

read PDF's -> Chunking -> Embeddings -> FAISS -> Retrieval -> LLM -> Answer and Sources


# Setup

In [ ]:
# Install packages
!pip install pymupdf
!pip install sentence-transformers
!pip install faiss-cpu
!pip install openai
!pip install langchain
!pip install langchain-community
!pip install -qU langchain-text-splitters

# Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import userdata
import os

BASE_PATH = "/content/drive/MyDrive/financial-risk-rag"
DATA_PATH = f"{BASE_PATH}/data/raw"
INDEX_PATH = f"{BASE_PATH}/index"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 100
TOP_K = 3

EMBED_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL = "gpt-4.1-mini"

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_RAG')

# Data Loading

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader


pdf_files = list(Path(DATA_PATH).glob("*.pdf"))

all_documents = []

# Iterate through all files in the folder
for filename in pdf_files:

        # Initialize the loader for each specific file
        loader = PyMuPDFLoader(str(filename))
        docs = loader.load()
        all_documents.extend(docs)
print("Total pages:",len(all_documents))

Total pages: 2070


# Document Processing
Document Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
#Split the document into chunks for processing
text_splitter = RecursiveCharacterTextSplitter(
      chunk_size = CHUNK_SIZE,
      chunk_overlap = CHUNK_OVERLAP
  )
# Split the documents into chunks and include source for each chunk

chunks =[]
for doc in all_documents:
  chunk_text = text_splitter.split_text(doc.page_content)
  for chunk in chunk_text:
    chunks.append({
        "source" : doc.metadata["source"],
        "page" : doc.metadata["page"],
        "chunk_text" : chunk
    })

print("Total chunks:",len(chunks))

Total chunks: 10040


# Embedding and Vector Index Creation

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# Load pre-trained embedding model
embedding_model = SentenceTransformer(EMBED_MODEL)

# Encode sentences
chunk_contents = [doc["chunk_text"] for doc in chunks]
embeddings = embedding_model.encode(chunk_contents,
                                    show_progress_bar=True)

# Initialize and build the index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/314 [00:00<?, ?it/s]

In [ ]:
import faiss
import pickle

faiss.write_index(index, f"{INDEX_PATH}/faiss.index")

with open(f"{INDEX_PATH}/chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

# Load chunks and FAISS index

In [ ]:
index = faiss.read_index(f"{INDEX_PATH}/faiss.index")

with open(f"{INDEX_PATH}/chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

In [ ]:
print("Total pdf's:",len(pdf_files))
print("Total pages:",len(all_documents))
print("Total chunks:",len(chunks))


Total pdf's: 7
Total pages: 2070
Total chunks: 10040


# Retrieval System

In [ ]:
# Prepare a function to retrieve and generate (RAG)
def retrieve_text(query,k=5):

  query_vector = embedding_model.encode([query]).astype("float32")
  distances, indices = index.search(query_vector,k)

  results=[]

  for i in indices[0]:
    results.append(chunks[i])
  return results

# RAG Pipeline

RAG Answer Generation

In [ ]:
# Connect to Open API and get the context for the retrieved text
from openai import OpenAI

client = OpenAI()

def get_context(query_result):

  context_parts = []
  for doc in query_result:
    source = doc["source"]
    page = doc["page"]

    context_parts.append(f"Source: {source}\nPage: {page} \n{doc["chunk_text"]}")
  return context_parts

In [ ]:
# Prepare a function to retrieve and generate (RAG)

def ask_query(query, k):

  # Retrieve relevant chunks from the FAISS index
  retrieved_chunks = retrieve_text(query,k)

  # Merge the content of retrieved documents into a single context string
  context_text = get_context(retrieved_chunks)

  # Define the prompt for the language model
  prompt = f"""
  You are a finanical risk assistant
  Only based on this context {context_text}
  Answer this question {query}
  Do not give a summary unless the question is asking for a summary

  Be conscise and to the point. Give a direct answer and short explanation

  Before answering, first determine if context directly answers the question. If not, return no information.

  Return:
  Just print answer to the question
  Provide KEY INSIGHTS in bullet points. Top 5 most relevant points.
  If relevant information is found, provide all the sources used to answer the question; Separate page numbers with ",". Add retreival Ranking
  If relevant information is not found, then for DOCUMENT SOURCES, write : None (no relevant context found). SKIP KEY INSIGHTS


  example:
  DOCUMENT SOURCES:
  Rank 1: ally-20241231.pdf | Page 34, 45



  Add a MODEL NOTE at the bottom that:
  Answer was generated using supporting documents only

  """

  # Call the LLM to generate a response
  model_response = client.responses.create(
      model = LLM_MODEL,
      input =prompt
  )

  return model_response.output_text,context_text

# Demo Function

In [ ]:
def demo_func(question,k):
  answer,sources = ask_query(question,k)
  print("="*100)
  print("QUESTION:")
  print(question)
  print("\nANSWER:")
  print(answer)
  print("="*100)

In [ ]:
# Chunk_size=800, chunk_ooverlap=100
demo_func("What factors increase credit losses?",k=3)

QUESTION:
What factors increase credit losses?

ANSWER:
The answer to the question:
Factors that increase credit losses include rising credit losses or leading indicators such as higher delinquencies, higher rates of nonperforming loans, higher bankruptcy rates, lower collateral values, elevated unemployment rates, changing market terms, missed or late payments by customers, and loan charge-offs preceded by missed payments or bankruptcies.

KEY INSIGHTS:
- Higher delinquencies indicate rising credit losses.
- Increased rates of nonperforming loans contribute to credit risk.
- Higher bankruptcy rates reduce recoveries and increase charge-offs.
- Decrease in collateral values can lead to higher credit losses.
- Elevated unemployment rates elevate default risk among borrowers.
- Missed payments are a direct precursor to loan charge-offs and default.

DOCUMENT SOURCES:
Rank 1: Capital_One_2024.pdf | Page 28

MODEL NOTE: Answer generated using supporting documents only


# Evaluation Questions

In [ ]:
test_questions = [
    "What factors increase credit losses?",
    "What econocmic risks exist?",
    "What risks affect borrower repayment ability?",
    "What risks affect liquidity?",
    "Which macro economic factors are related to credit risks?",
    "Which are the similar economic risks discussed across all documents?",
    "What risks appear across multiple documents",
    "Which neural network architecture is used in each document?",
    "What is the exact model accuracy of the company's risk model?"
]

In [ ]:
for question in test_questions:
  demo_func(question,k=3)

QUESTION:
What factors increase credit losses?

ANSWER:
Factors that increase credit losses include rising credit losses or leading indicators such as higher delinquencies, higher rates of nonperforming loans, higher bankruptcy rates, lower collateral values, elevated unemployment rates, changing market terms, missed payments, defaults, and delinquencies by customers.

KEY INSIGHTS:
- Higher delinquencies and nonperforming loans contribute to increased credit losses.
- Elevated bankruptcy rates increase likelihood of loan charge-offs.
- Lower collateral values reduce recovery from loans.
- Elevated unemployment rates negatively impact borrower repayment ability.
- Missed payments and defaults are direct precursors to loan charge-offs.

DOCUMENT SOURCES:
Rank 1: Capital_One_2024.pdf | Page 28

MODEL NOTE:
Answer was generated using supporting documents only
QUESTION:
What econocmic risks exist?

ANSWER:
Economic risks identified include:

- Credit risk: loss from obligors failing to mee

Evaluation Summary:

The system was tested on 9 questions which cover multi-document, factual and unsupported queries.

Results showed:


*   Grounded responses
*   Safe handling of unsupported queries
*   Relevant document retrieval
*   Multi-document retrieval





# Future Improvements